# Deteccion de objetos con MediaPipe

Primera iteracion de la prueba de concepto. Este cuaderno funciona en Google Colab y en Jupyter local. Permite cargar una imagen, detectar objetos conocidos y mostrar rectangulos con etiquetas y porcentajes de confianza.

## 1. Instalar las bibliotecas

MediaPipe ejecuta el modelo de vision artificial y OpenCV permite leer y dibujar sobre la imagen.

In [ ]:
!pip install -q mediapipe opencv-python

## 2. Importar las herramientas

Importar significa hacer disponibles las funciones de las bibliotecas dentro del cuaderno.

In [ ]:
from pathlib import Path
import urllib.request

import cv2
import mediapipe as mp
import numpy as np

# Colab ofrece un selector de archivos propio. En Jupyter local se usa una ruta.
try:
    from google.colab import files
    from google.colab.patches import cv2_imshow
    EN_COLAB = True
    print('Entorno detectado: Google Colab')
except ImportError:
    from IPython.display import Image, display
    EN_COLAB = False
    print('Entorno detectado: Jupyter local')

## 3. Descargar el modelo preentrenado

EfficientDet-Lite0 ya fue entrenado para reconocer categorias generales. El archivo se descarga solo si todavia no existe en la sesion de Colab.

In [ ]:
MODELO_URL = (
    'https://storage.googleapis.com/mediapipe-models/object_detector/'
    'efficientdet_lite0/float32/1/efficientdet_lite0.tflite'
)
MODELO_RUTA = Path('efficientdet_lite0.tflite')

if not MODELO_RUTA.exists():
    print('Descargando el modelo...')
    urllib.request.urlretrieve(MODELO_URL, MODELO_RUTA)

print('Modelo listo:', MODELO_RUTA)

## 4. Configurar el detector

El umbral `0.45` significa que se descartan detecciones con menos de 45 % de confianza. `max_results=5` limita la salida a cinco objetos.

In [ ]:
opciones = mp.tasks.vision.ObjectDetectorOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path=str(MODELO_RUTA.resolve())
    ),
    running_mode=mp.tasks.vision.RunningMode.IMAGE,
    score_threshold=0.45,
    max_results=5,
)

detector = mp.tasks.vision.ObjectDetector.create_from_options(opciones)
print('Detector listo')

## 5. Funciones para detectar y dibujar

OpenCV lee colores en orden BGR. Antes de enviar la imagen a MediaPipe se convierte a RGB. El modelo devuelve una caja, una categoria y una confianza por cada deteccion.

In [ ]:
TRADUCCIONES = {
    'person': 'persona',
    'cup': 'taza',
    'cell phone': 'telefono',
    'bottle': 'botella',
    'chair': 'silla',
    'book': 'libro',
    'laptop': 'computadora portatil',
    'keyboard': 'teclado',
    'mouse': 'mouse',
}

def leer_imagen(ruta):
    datos = np.fromfile(ruta, dtype=np.uint8)
    imagen = cv2.imdecode(datos, cv2.IMREAD_COLOR)
    if imagen is None:
        raise ValueError('No se pudo leer la imagen seleccionada')
    return imagen

def detectar_y_dibujar(imagen_bgr):
    imagen_rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
    imagen_mp = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=imagen_rgb,
    )
    resultado = detector.detect(imagen_mp)
    imagen_anotada = imagen_bgr.copy()

    for deteccion in resultado.detections:
        categoria = deteccion.categories[0]
        nombre = TRADUCCIONES.get(
            categoria.category_name, categoria.category_name
        )
        confianza = categoria.score
        caja = deteccion.bounding_box
        inicio = (caja.origin_x, caja.origin_y)
        fin = (
            caja.origin_x + caja.width,
            caja.origin_y + caja.height,
        )

        cv2.rectangle(imagen_anotada, inicio, fin, (40, 200, 40), 3)
        texto = f'{nombre}: {confianza:.0%}'
        texto_y = max(25, caja.origin_y - 10)
        cv2.putText(
            imagen_anotada, texto, (caja.origin_x, texto_y),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (40, 200, 40), 2
        )
        print(f'- {nombre}: {confianza:.0%}')

    return imagen_anotada, resultado

## 6. Cargar y procesar una imagen

En Colab aparecera el boton **Elegir archivos**. En Jupyter local, coloca una imagen llamada `foto.jpg` junto al notebook o cambia el valor de `RUTA_IMAGEN_LOCAL`. Puedes volver a ejecutar esta celda para probar otra imagen.

In [ ]:
RUTA_IMAGEN_LOCAL = Path('foto.jpg')  # Cambia esta ruta si usas Jupyter local

if EN_COLAB:
    archivos = files.upload()
    if not archivos:
        raise RuntimeError('No se selecciono ninguna imagen')
    ruta_imagen = Path(next(iter(archivos)))
else:
    ruta_imagen = RUTA_IMAGEN_LOCAL
    if not ruta_imagen.is_file():
        raise FileNotFoundError(
            f'No existe {ruta_imagen}. Copia foto.jpg junto al notebook '
            'o cambia RUTA_IMAGEN_LOCAL.'
        )

imagen = leer_imagen(ruta_imagen)
print('Objetos detectados:')
imagen_resultado, resultado = detectar_y_dibujar(imagen)

ruta_salida = 'resultado_deteccion.jpg'
cv2.imwrite(ruta_salida, imagen_resultado)
if EN_COLAB:
    cv2_imshow(imagen_resultado)
else:
    display(Image(filename=ruta_salida))
print('Resultado guardado en:', ruta_salida)

## 7. Experimentos sugeridos

- Prueba una taza, un telefono y una persona.
- Cambia `score_threshold` y vuelve a crear el detector.
- Observa que un umbral bajo produce mas detecciones y posibles errores.
- Prueba cambios de distancia e iluminacion.